# Why 1000 Scans Didn't Help: The Truth About Signal Averaging
### 3.6 — Signal Averaging and the √N Rule: When More Scans Help and When They Don't

"Just average more scans" is the most common signal-to-noise move in the lab — and
it's right, under one specific condition. Co-adding scans improves your
signal-to-noise ratio in proportion to **√N**: four scans halve the noise, a hundred
cut it tenfold. But that improvement assumes a particular kind of noise and a
particular kind of sample, and when those assumptions break, more scans stop
helping — or quietly mislead you into reporting a spectrum of a sample that no longer
exists.

This lesson is less about code than judgement. The averaging itself is one line; the
skill is knowing **when it works, when it plateaus, and when it lies.**

> **One idea to hold onto:** averaging N scans improves SNR by **√N** — but only for
> random, uncorrelated noise on a *stable* sample. Drift, correlated noise, or a
> changing sample break the rule, and more scans can mislead instead of help.

**By the end of this notebook you will be able to:**

1. Co-add replicate scans and measure the √N improvement under ideal white noise.
2. See averaging stop helping (drift) and start lying (a changing sample).
3. Decide how many scans are actually worth acquiring — a measurement judgement, not a default.

## 1. One noisy scan, and the plan

We measure a single absorbance band buried in noise. Averaging works by exploiting a
simple asymmetry: the **signal repeats** in every scan and adds up coherently, while
**random noise** differs each scan and partly cancels. We'll quantify "how noisy" with
the RMS deviation of the averaged spectrum from the known true band, measured in a
flat, peak-free region.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (6.5, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})

wl = np.linspace(580, 620, 401)                  # a small window around one band
true_band = np.exp(-((wl - 600) ** 2) / (2 * 3.0 ** 2))   # unit-height band at 600 nm
flat = wl <= 592                                  # a peak-free region for measuring noise
NOISE = 0.05                                      # per-scan white-noise sigma (AU)

rng = np.random.default_rng(0)
one_scan = true_band + rng.normal(0, NOISE, size=wl.size)

def rms_error(averaged):
    "RMS deviation of an averaged spectrum from the truth, in the flat region."
    return np.sqrt(np.mean((averaged[flat] - true_band[flat]) ** 2))

print("single-scan RMS noise:", round(rms_error(one_scan), 4), "AU")
print("band height          :", round(true_band.max(), 3), "AU")
print("single-scan SNR      :", round(true_band.max() / rms_error(one_scan), 1))

## 2. The ideal case: averaging works, and it works as √N

We co-add up to 64 independent scans of the **same, stable** sample with pure white
noise, and measure the noise after averaging the first N. The prediction is
`sigma_N = sigma_1 / √N`.

In [ ]:
N_MAX = 64
# A stack of independent scans: same true band, fresh white noise each row.
scans_ideal = true_band + rng.normal(0, NOISE, size=(N_MAX, wl.size))

Ns = np.array([1, 2, 4, 8, 16, 32, 64])
rms_ideal = np.array([rms_error(scans_ideal[:n].mean(axis=0)) for n in Ns])
theory = NOISE / np.sqrt(Ns)                      # the sqrt(N) prediction

print(" N   measured RMS   sqrt(N) theory   SNR")
for n, meas, th in zip(Ns, rms_ideal, theory):
    print(f"{n:3d}    {meas:.4f}        {th:.4f}       {true_band.max()/meas:5.1f}")

fig, ax = plt.subplots()
ax.loglog(Ns, rms_ideal, "o-", color="#1a73e8", label="measured")
ax.loglog(Ns, theory, "--", color="#5f6368", label=r"$\sigma_1/\sqrt{N}$ (theory)")
ax.set_xlabel("number of scans averaged, N")
ax.set_ylabel("RMS noise (AU)")
ax.set_title("Ideal white noise: averaging follows the √N rule")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

The measured points sit right on the √N line. Note what √N actually buys: going from
1 to 4 scans halves the noise, but halving it *again* takes 16 scans, and again, 64.
**The cost of signal-to-noise grows quadratically** — to improve SNR by 10× you pay
100× the acquisition time. That diminishing return is the first thing to weigh.

## 3. Why √N — in one sentence

Averaging N independent measurements of the same quantity reduces the standard
deviation of the result by √N, because the signal adds **coherently** (N copies of
the same band → N× taller) while independent noise adds **incoherently** (random
deviations partly cancel → only √N× larger). The ratio improves as N/√N = √N. The
load-bearing word is **independent** — and that's exactly what the next two cases break.

## 4. When averaging stops helping: drift

White noise is independent from scan to scan. **Drift is not.** If the baseline
creeps during a long acquisition — a warming lamp, a settling detector — each scan
carries a little more offset than the last. The random noise still falls as √N, but
the drift is *systematic*: averaging can't cancel it, and as it accumulates it
eventually dominates. More scans then make the result **worse**.

In [ ]:
DRIFT_PER_SCAN = 0.0015                            # baseline creep added each scan (AU)
drift = DRIFT_PER_SCAN * np.arange(N_MAX)          # scan 0, 1, 2, ... drifts upward
scans_drift = true_band + drift[:, None] + rng.normal(0, NOISE, size=(N_MAX, wl.size))

rms_drift = np.array([rms_error(scans_drift[:n].mean(axis=0)) for n in Ns])

fig, ax = plt.subplots()
ax.loglog(Ns, rms_ideal, "o-", color="#1a73e8", label="stable sample (ideal)")
ax.loglog(Ns, rms_drift, "s-", color="#ea4335", label="with baseline drift")
ax.loglog(Ns, theory, "--", color="#5f6368", label=r"$\sqrt{N}$ ideal")
ax.set_xlabel("number of scans averaged, N")
ax.set_ylabel("RMS noise (AU)")
ax.set_title("Drift breaks √N: noise falls, then accumulated drift takes over")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

best = Ns[np.argmin(rms_drift)]
print("with drift, the best result is at N =", int(best),
      "scans -- averaging MORE makes it worse.")

The drift curve tracks the ideal one at first, bottoms out, then climbs: past a
point, every extra scan adds more drift than it removes noise. There is an **optimal
number of scans**, and it's set by how fast your instrument drifts, not by how clean
you'd like the data to look. (Correlated noise like mains hum behaves the same way —
it's not independent, so it doesn't average away either.)

## 5. When averaging lies: a changing sample

The most dangerous case looks perfectly clean. Averaging assumes every scan measures
the *same* sample. If the sample is changing during acquisition — photobleaching,
degrading, reacting — then each scan measures a different thing, and the average is
of a sample that existed at no single moment. Here the analyte decays as we acquire.

In [ ]:
TAU = 30.0                                         # decay constant, in scans
amplitude = np.exp(-np.arange(N_MAX) / TAU)        # band shrinks scan to scan
scans_changing = amplitude[:, None] * true_band + rng.normal(0, NOISE, size=(N_MAX, wl.size))

i_peak = np.argmax(true_band)
heights = np.array([scans_changing[:n].mean(axis=0)[i_peak] for n in Ns])

print(" N    averaged peak height (true initial = 1.000)")
for n, h in zip(Ns, heights):
    print(f"{n:3d}    {h:.3f}")

fig, ax = plt.subplots()
ax.plot(wl, scans_changing[0], lw=0.8, color="#dadce0", label="scan 1 (noisy)")
ax.plot(wl, scans_changing.mean(axis=0), lw=1.8, color="#ea4335",
        label=f"average of {N_MAX} scans")
ax.plot(wl, true_band, lw=1.8, color="#1a73e8", label="true initial band")
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("absorbance (AU)")
ax.set_title("A decaying sample: the average is cleaner but WRONG (too low)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

The 64-scan average is beautifully smooth — and reports a band height far below the
true initial value, because it blends the bright early scans with the faded late
ones. This is the trap: **the noise looks great, so the result looks trustworthy**,
while the averaging has quietly buried a real, time-dependent change. No amount of
extra scanning fixes it; it makes it worse. The fix is to recognize the sample is
changing (e.g. the scans aren't reproducible — the test from 3.1) and to measure
fast, or model the kinetics, instead of averaging across them.

## 6. The judgement: how many scans?

Signal averaging is a tool with a domain of validity, and using it well is a
measurement decision:

- **Diminishing returns.** SNR grows as √N, so cost grows as N. Decide the SNR you
  actually need and stop — chasing another factor of two rarely justifies 4× the time.
- **Stability budget.** Average only as long as the instrument and sample hold still.
  If drift sets in at ~20 scans, averaging 1000 is worse than averaging 20.
- **Check reproducibility.** If early and late scans disagree beyond noise, the sample
  or instrument is changing and averaging is the wrong tool — measure faster or model
  the change. (A formal way to find the stability limit is the Allan variance, a good
  follow-up.)
- **Random noise only.** Averaging beats down *random* noise. It does nothing for
  systematic error, drift, or correlated interference — those need correction, not repetition.

## Key Takeaways

- Averaging N scans improves SNR by **√N** — but only for random, independent noise on
  a stable sample.
- The cost is quadratic: **10× SNR costs 100× the scans.** Decide the SNR you need.
- **Drift and correlated noise** aren't independent, so they don't average out — there
  is an optimal N, beyond which more scans hurt.
- A **changing sample** makes the average clean but wrong; good-looking noise is not
  the same as a trustworthy result.

## Practical Checklist

- [ ] Estimate the SNR you actually need before choosing N.
- [ ] Compare early vs late scans; if they differ beyond noise, stop averaging.
- [ ] Watch for baseline drift — it sets the useful averaging limit, not the clock.
- [ ] Remember averaging fixes random noise only; correct drift and systematics separately.

## Common Mistakes

- Assuming more scans are always better — past the drift limit they degrade the result.
- Trusting a clean averaged spectrum without checking the sample was stable.
- Trying to average away mains hum, drift, or other correlated/systematic effects.
- Quoting an SNR gain from N without noting the (quadratic) acquisition-time cost.

## Reporting Guidance

- Report the **number of co-added scans** and that sample/instrument stability was
  checked over the acquisition — averaging is part of how the spectrum was produced.

## Next Lesson

**3.7 — Frequency Domain: A Practical Look at the FFT.** With clean, honestly
averaged spectra in hand, next we change axes entirely — from *when* to *how
often* — to find periodic interference (pumps, mains hum, daily rhythms) that
hides in the time trace.